# Pay Equity Analysis

## Purpose
Identify and quantify unexplained pay gaps across demographic groups using regression-controlled analysis.

## Key Metrics
- **Raw Pay Gap**: Unadjusted difference in average compensation
- **Adjusted Pay Gap**: Difference after controlling for legitimate factors (level, performance, tenure)
- **Unexplained Variance**: Portion of pay variance attributable to demographics
- **Compa-Ratio**: Actual salary vs market midpoint (target: 0.95-1.05)

## Research Foundation
- Oaxaca-Blinder decomposition methodology (1973)
- EEOC guidelines on pay discrimination
- Regression-controlled analysis (Castilla & Benard, 2010)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Generate data if it doesn't exist
if not os.path.exists('../data/employees_compensation.csv'):
    print("Generating sample data...")
    exec(open('../data/generate_sample_data.py').read())
else:
    print("Loading existing data...")

# Load datasets
employees = pd.read_csv('../data/employees_compensation.csv')

print(f"\nLoaded {len(employees)} employee compensation records")
print(f"Total payroll: ${employees['base_salary'].sum():,.0f}")
print(f"Total compensation (all rewards): ${employees['total_compensation'].sum():,.0f}")

## 1. Descriptive Pay Statistics

Start with raw (unadjusted) pay comparisons across demographic groups.

In [ ]:
print(f"{'='*80}")
print(f"RAW PAY STATISTICS")
print(f"{'='*80}")

# Overall statistics
print(f"\nOverall Compensation:")
print(f"  Mean base salary:   ${employees['base_salary'].mean():,.0f}")
print(f"  Median base salary: ${employees['base_salary'].median():,.0f}")
print(f"  Std deviation:      ${employees['base_salary'].std():,.0f}")
print(f"  Min:                ${employees['base_salary'].min():,.0f}")
print(f"  Max:                ${employees['base_salary'].max():,.0f}")

# Gender pay gap
print(f"\n{'='*80}")
print(f"GENDER PAY ANALYSIS")
print(f"{'='*80}")

gender_pay = employees.groupby('gender')['base_salary'].agg(['mean', 'median', 'count'])
gender_pay = gender_pay.round(0)
print(gender_pay)

male_avg = gender_pay.loc['Male', 'mean']
female_avg = gender_pay.loc['Female', 'mean']
gender_gap = (male_avg - female_avg) / male_avg * 100

print(f"\nRaw gender pay gap: {gender_gap:.1f}%")
print(f"  Male employees earn ${male_avg - female_avg:,.0f} more on average")

if gender_gap > 5:
    print(f"  ⚠ GAP EXCEEDS 5% - Requires detailed investigation")

# Race pay analysis
print(f"\n{'='*80}")
print(f"RACE PAY ANALYSIS")
print(f"{'='*80}")

race_pay = employees.groupby('race')['base_salary'].agg(['mean', 'median', 'count'])
race_pay = race_pay.sort_values('mean', ascending=False).round(0)
print(race_pay)

# Calculate gaps relative to highest-paid group
highest_paid_race = race_pay.index[0]
highest_avg = race_pay.iloc[0]['mean']

print(f"\nPay gaps relative to {highest_paid_race} (highest-paid group):")
for race in race_pay.index[1:]:
    race_avg = race_pay.loc[race, 'mean']
    gap = (highest_avg - race_avg) / highest_avg * 100
    print(f"  {race:20s}: {gap:.1f}% gap (${highest_avg - race_avg:,.0f} difference)")
    if gap > 7:
        print(f"    ⚠ GAP EXCEEDS 7% - Flag for investigation")

print(f"\n{'='*80}")

# Visualize pay distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gender pay distribution
for gender in ['Male', 'Female']:
    data = employees[employees['gender'] == gender]['base_salary']
    ax1.hist(data, bins=30, alpha=0.6, label=gender, edgecolor='black')

ax1.set_xlabel('Base Salary ($)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Employees', fontsize=11, fontweight='bold')
ax1.set_title('Base Salary Distribution by Gender', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Race pay comparison
race_data = [employees[employees['race'] == race]['base_salary'].values 
             for race in race_pay.index if len(employees[employees['race'] == race]) > 10]
race_labels = [race for race in race_pay.index if len(employees[employees['race'] == race]) > 10]

bp = ax2.boxplot(race_data, labels=race_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax2.set_ylabel('Base Salary ($)', fontsize=11, fontweight='bold')
ax2.set_title('Base Salary Distribution by Race', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Regression-Controlled Pay Equity Analysis

Control for legitimate factors (level, performance, tenure) to isolate unexplained pay variance.

In [ ]:
# Prepare data for regression
reg_data = employees.copy()

# Encode categorical variables
reg_data['level_encoded'] = LabelEncoder().fit_transform(reg_data['level'])
reg_data['dept_encoded'] = LabelEncoder().fit_transform(reg_data['department'])
reg_data['location_encoded'] = LabelEncoder().fit_transform(reg_data['location'])

# Create dummy variables for demographics
gender_dummies = pd.get_dummies(reg_data['gender'], prefix='gender', drop_first=True)
race_dummies = pd.get_dummies(reg_data['race'], prefix='race', drop_first=True)

reg_data = pd.concat([reg_data, gender_dummies, race_dummies], axis=1)

# Define feature sets
control_features = ['level_encoded', 'performance_rating', 'tenure_months']
location_features = ['location_encoded']
dept_features = ['dept_encoded']
gender_features = [col for col in reg_data.columns if col.startswith('gender_')]
race_features = [col for col in reg_data.columns if col.startswith('race_')]

# Target variable
y = reg_data['base_salary']

print(f"{'='*80}")
print(f"REGRESSION-CONTROLLED PAY EQUITY ANALYSIS")
print(f"{'='*80}")

# Model 1: Control variables only (legitimate factors)
X_control = reg_data[control_features + location_features + dept_features]
model_control = LinearRegression()
model_control.fit(X_control, y)
r2_control = model_control.score(X_control, y)

print(f"\nModel 1: Legitimate Factors Only")
print(f"  Features: Level, Performance, Tenure, Location, Department")
print(f"  R² = {r2_control:.3f} ({r2_control*100:.1f}% of variance explained)")

# Model 2: Add demographics
X_full = reg_data[control_features + location_features + dept_features + gender_features + race_features]
model_full = LinearRegression()
model_full.fit(X_full, y)
r2_full = model_full.score(X_full, y)

print(f"\nModel 2: Legitimate Factors + Demographics")
print(f"  Features: Level, Performance, Tenure, Location, Department + Gender, Race")
print(f"  R² = {r2_full:.3f} ({r2_full*100:.1f}% of variance explained)")

# Calculate unexplained variance
r2_increase = r2_full - r2_control
unexplained_variance = r2_increase * 100

print(f"\n📊 PAY EQUITY ASSESSMENT:")
print(f"  Unexplained variance from demographics: {unexplained_variance:.2f}%")
print(f"  Threshold for concern: 2.0%")

if unexplained_variance > 2.0:
    print(f"  ⚠ ALERT: Demographic factors explain >{unexplained_variance:.1f}% of pay variance")
    print(f"           This suggests systematic pay inequity requiring investigation")
elif unexplained_variance > 1.0:
    print(f"  → MONITOR: Modest demographic effect, continue tracking")
else:
    print(f"  ✓ PASS: Demographics have minimal impact on pay (<1%)")

print(f"\n{'='*80}")

# Extract coefficients for demographic variables
feature_names = (control_features + location_features + dept_features + 
                 gender_features + race_features)
coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model_full.coef_
})

# Filter to demographic variables
demo_coefs = coefficients[coefficients['Feature'].str.contains('gender_|race_')].copy()
demo_coefs['Coefficient'] = demo_coefs['Coefficient'].round(0)
demo_coefs = demo_coefs.sort_values('Coefficient')

print(f"\nDemographic Pay Adjustments (controlled for level, performance, tenure):")
print(demo_coefs.to_string(index=False))
print(f"\nInterpretation: Negative values = systematic underpayment relative to baseline")

# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['red' if x < -1000 else 'orange' if x < 0 else 'green' 
          for x in demo_coefs['Coefficient']]

bars = ax.barh(demo_coefs['Feature'], demo_coefs['Coefficient'], 
               color=colors, alpha=0.7, edgecolor='black')

ax.axvline(x=0, color='black', linewidth=1)
ax.axvline(x=-2000, color='red', linestyle='--', alpha=0.5, label='Significant Gap ($2K)')
ax.set_xlabel('Pay Adjustment ($)', fontsize=12, fontweight='bold')
ax.set_title('Demographic Pay Gaps (Regression-Controlled)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(demo_coefs.iterrows()):
    x_pos = row['Coefficient'] + (-500 if row['Coefficient'] < 0 else 500)
    ha = 'right' if row['Coefficient'] < 0 else 'left'
    ax.text(x_pos, i, f"${row['Coefficient']:,.0f}", 
            va='center', ha=ha, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Pay-for-Performance Correlation

Assess whether pay appropriately reflects performance ratings.

In [ ]:
# Analyze pay vs performance relationship
print(f"{'='*80}")
print(f"PAY-FOR-PERFORMANCE CORRELATION")
print(f"{'='*80}")

# Overall correlation
corr_overall, pval_overall = stats.pearsonr(employees['performance_rating'], 
                                             employees['base_salary'])

print(f"\nOverall correlation: r = {corr_overall:.3f}")
if pval_overall < 0.05:
    print(f"  ✓ SIGNIFICANT (p<0.05): Pay correlates with performance")
else:
    print(f"  ⚠ NOT SIGNIFICANT: Pay does not correlate with performance")

# Correlation by level (to control for level effects)
print(f"\nPay-Performance Correlation by Level:")
for level in ['IC3', 'IC4', 'Manager', 'Senior Manager', 'Director']:
    level_data = employees[employees['level'] == level]
    if len(level_data) > 10:
        corr, pval = stats.pearsonr(level_data['performance_rating'], 
                                    level_data['base_salary'])
        sig = "✓" if pval < 0.05 else "×"
        print(f"  {level:20s}: r={corr:6.3f} {sig}")

print(f"\n{'='*80}")

# Average pay by performance rating
perf_pay = employees.groupby(employees['performance_rating'].round())['base_salary'].agg(['mean', 'count'])
perf_pay = perf_pay.round(0)
perf_pay.columns = ['Avg Salary', 'Count']

print(f"\nAverage Salary by Performance Rating:")
print(perf_pay)

# Calculate pay spread
if 5.0 in perf_pay.index and 1.0 in perf_pay.index:
    top_pay = perf_pay.loc[5.0, 'Avg Salary']
    bottom_pay = perf_pay.loc[1.0, 'Avg Salary']
    pay_spread = (top_pay - bottom_pay) / bottom_pay * 100
    print(f"\nPay spread (top vs bottom performers): {pay_spread:.1f}%")
    if pay_spread < 20:
        print(f"  ⚠ LOW DIFFERENTIATION: Consider stronger pay-for-performance")
    else:
        print(f"  ✓ ADEQUATE DIFFERENTIATION: Pay reflects performance differences")

# Visualize pay vs performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
ax1.scatter(employees['performance_rating'], employees['base_salary'], 
           alpha=0.4, s=50, edgecolors='black', linewidth=0.5)

# Add trend line
z = np.polyfit(employees['performance_rating'], employees['base_salary'], 1)
p = np.poly1d(z)
x_trend = np.linspace(1, 5, 100)
ax1.plot(x_trend, p(x_trend), "r--", linewidth=2, label=f'r={corr_overall:.3f}')

ax1.set_xlabel('Performance Rating (1-5)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Base Salary ($)', fontsize=11, fontweight='bold')
ax1.set_title('Pay vs Performance Relationship', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot by rating
perf_ratings = sorted(employees['performance_rating'].round().unique())
perf_data = [employees[employees['performance_rating'].round() == rating]['base_salary'].values 
             for rating in perf_ratings]

bp = ax2.boxplot(perf_data, labels=perf_ratings, patch_artist=True)
for i, patch in enumerate(bp['boxes']):
    # Color gradient from red to green
    color = plt.cm.RdYlGn(i / len(perf_ratings))
    patch.set_facecolor(color)

ax2.set_xlabel('Performance Rating', fontsize=11, fontweight='bold')
ax2.set_ylabel('Base Salary ($)', fontsize=11, fontweight='bold')
ax2.set_title('Salary Distribution by Performance Rating', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compa-Ratio Analysis

Identify employees paid significantly above or below market midpoint.

In [ ]:
print(f"{'='*80}")
print(f"COMPA-RATIO ANALYSIS")
print(f"{'='*80}")

# Overall compa-ratio statistics
print(f"\nCompa-Ratio Statistics:")
print(f"  Mean:   {employees['compa_ratio'].mean():.3f}")
print(f"  Median: {employees['compa_ratio'].median():.3f}")
print(f"  Std Dev: {employees['compa_ratio'].std():.3f}")
print(f"  Target range: 0.95 - 1.05")

# Categorize compa-ratios
employees['compa_category'] = pd.cut(
    employees['compa_ratio'],
    bins=[0, 0.90, 0.95, 1.05, 1.10, 2.0],
    labels=['Severely Underpaid', 'Underpaid', 'On Target', 'Overpaid', 'Severely Overpaid']
)

compa_dist = employees['compa_category'].value_counts()
compa_pcts = (compa_dist / len(employees) * 100).round(1)

print(f"\nCompa-Ratio Distribution:")
for category in ['Severely Underpaid', 'Underpaid', 'On Target', 'Overpaid', 'Severely Overpaid']:
    if category in compa_dist.index:
        count = compa_dist[category]
        pct = compa_pcts[category]
        marker = "⚠" if category in ['Severely Underpaid', 'Severely Overpaid'] else " "
        print(f"  {marker} {category:20s}: {count:3d} employees ({pct:5.1f}%)")

# Identify outliers
severely_underpaid = employees[employees['compa_ratio'] < 0.90]
severely_overpaid = employees[employees['compa_ratio'] > 1.10]

print(f"\n⚠ OUTLIERS REQUIRING REVIEW:")
print(f"  Severely underpaid (<0.90): {len(severely_underpaid)} employees")
print(f"  Severely overpaid (>1.10):  {len(severely_overpaid)} employees")

if len(severely_underpaid) > 0:
    print(f"\n  Top 5 most underpaid employees:")
    underpaid_top5 = severely_underpaid.nsmallest(5, 'compa_ratio')[[
        'employee_id', 'level', 'department', 'compa_ratio', 'base_salary', 'market_50th'
    ]]
    for _, row in underpaid_top5.iterrows():
        gap = row['market_50th'] - row['base_salary']
        print(f"    ID {row['employee_id']}: {row['level']:15s} {row['department']:15s} "
              f"Compa={row['compa_ratio']:.3f} (${gap:,.0f} below market)")

if len(severely_overpaid) > 0:
    print(f"\n  Top 5 most overpaid employees:")
    overpaid_top5 = severely_overpaid.nlargest(5, 'compa_ratio')[[
        'employee_id', 'level', 'department', 'compa_ratio', 'base_salary', 'market_50th'
    ]]
    for _, row in overpaid_top5.iterrows():
        excess = row['base_salary'] - row['market_50th']
        print(f"    ID {row['employee_id']}: {row['level']:15s} {row['department']:15s} "
              f"Compa={row['compa_ratio']:.3f} (${excess:,.0f} above market)")

print(f"\n{'='*80}")

# Visualize compa-ratio distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
ax1.hist(employees['compa_ratio'], bins=30, color='skyblue', alpha=0.7, edgecolor='black')
ax1.axvline(x=0.95, color='orange', linestyle='--', linewidth=2, label='Lower Bound (0.95)')
ax1.axvline(x=1.00, color='green', linestyle='-', linewidth=2, label='Market Midpoint (1.00)')
ax1.axvline(x=1.05, color='orange', linestyle='--', linewidth=2, label='Upper Bound (1.05)')
ax1.set_xlabel('Compa-Ratio', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Employees', fontsize=11, fontweight='bold')
ax1.set_title('Compa-Ratio Distribution', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Category counts
categories = ['Severely\nUnderpaid', 'Underpaid', 'On Target', 'Overpaid', 'Severely\nOverpaid']
category_map = {
    'Severely Underpaid': 'Severely\nUnderpaid',
    'Underpaid': 'Underpaid',
    'On Target': 'On Target',
    'Overpaid': 'Overpaid',
    'Severely Overpaid': 'Severely\nOverpaid'
}

counts = [compa_dist.get(list(category_map.keys())[i], 0) for i in range(len(categories))]
colors_cat = ['darkred', 'red', 'green', 'orange', 'darkred']

bars = ax2.bar(categories, counts, color=colors_cat, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Number of Employees', fontsize=11, fontweight='bold')
ax2.set_title('Compa-Ratio Categories', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar, count in zip(bars, counts):
    if count > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., count + 2,
                f'{count}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Raw gaps require context**: Unadjusted pay differences may reflect legitimate factors like level and performance
2. **Regression controls isolate bias**: Unexplained variance >2% suggests systematic inequity
3. **Pay-for-performance matters**: Strong correlation ensures merit-based compensation
4. **Compa-ratios identify outliers**: Employees significantly outside 0.95-1.05 range need review

**Recommended Next Steps**:
- Investigate root causes of demographic pay gaps identified in regression analysis
- Review individual cases of severely underpaid employees for immediate adjustment
- Ensure pay-for-performance differentiation is adequate across all levels
- Implement regular (annual) pay equity audits to maintain compliance